# Análise de Sentimentos e Sumarização Inteligente de Avaliações de Produtos E-commerce com Transformers

## Etapa 1: Carregamento e Pré-processamento dos Dados

### Dataset Escolhido: ptbr-sentiment-analysis-datasets (B2W)

Utilizaremos o dataset **ptbr-sentiment-analysis-datasets** do Kaggle, que contém avaliações reais de produtos brasileiros de fontes como B2W, Buscapé e Olist. Este dataset é ideal porque:

- **Fonte**: Avaliações reais de e-commerce brasileiro
- **Língua**: Português brasileiro nativo
- **Estrutura**: Já pré-processado com colunas polarity (0/1) e rating (1-5)
- **Volume**: ~130k avaliações da B2W (subset principal)
- **Disponibilidade**: Download via KaggleHub

**Estrutura do Dataset**:
- `review_text`: Texto original da avaliação
- `review_text_processed`: Texto pré-processado (lowercase, sem acentos)
- `polarity`: Sentimento binário (0=negativo, 1=positivo)
- `rating`: Classificação em estrelas (1-5)

**Mapeamento de Sentimentos**:
- polarity 0 → Negativo (0)
- polarity 1 → Positivo (1)

### Explicação Técnica do Pré-processamento

**Dataset B2W**: 
- Avaliações reais de e-commerce brasileiro
- Dados já pré-processados (tokenização, lowercase, remoção de acentos)
- Distribuição balanceada entre classes positivas/negativas

**Mapeamento de Sentimentos**: 
- polarity binária (0/1) mapeada diretamente para labels (0/1)
- Adequado para classificação binária de sentimentos
- Simplifica o problema comparado à classificação 3 classes

**Tokenização com BERTimbau**:
- Modelo BERT específico para português brasileiro
- Melhor performance em pt-BR comparado ao XLM-RoBERTa multilingual
- `max_length=256`: Adequado para reviews que podem ser mais longas

**Data Collator**: 
- Padding dinâmico otimiza uso de memória
- Processamento em batches acelera treinamento
- Attention mask identifica tokens reais vs padding

**Divisão dos Dados**:
- 80% treino, 10% validação, 10% teste
- Mantém distribuição original das classes
- Validação para tuning de hiperparâmetros durante treinamento

### Download e Carregamento do Dataset

In [ ]:
# Download do dataset via KaggleHub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fredericods/ptbr-sentiment-analysis-datasets")

print("Path to dataset files:", path)

# Carregamento do dataset B2W (principal)
import pandas as pd
import os

# Carregar o arquivo B2W
b2w_path = os.path.join(path, "b2w.csv")
df = pd.read_csv(b2w_path)

print("Dataset carregado com sucesso!")
print(f"Tamanho do dataset: {len(df)} avaliações")
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nPrimeiras 5 linhas:")
print(df.head())

print("\nDistribuição de polaridade:")
print(df['polarity'].value_counts())

print("\nDistribuição de rating:")
print(df['rating'].value_counts())

### Conversão para Dataset Hugging Face e Mapeamento

In [ ]:
# Converter para Dataset do Hugging Face
from datasets import Dataset, DatasetDict

# Usar apenas colunas necessárias
df_processed = df[['review_text_processed', 'polarity']].copy()
df_processed = df_processed.rename(columns={'review_text_processed': 'text', 'polarity': 'label'})

# Remover valores nulos se houver
df_processed = df_processed.dropna()

print(f"Dataset processado: {len(df_processed)} avaliações")
print("Distribuição das classes:")
print(df_processed['label'].value_counts())

# Converter para Dataset Hugging Face
dataset = Dataset.from_pandas(df_processed)

# Dividir em train/validation/test
# Usando proporções: 80% treino, 10% validação, 10% teste
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

dataset_final = DatasetDict({
    'train': train_test_split['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

print("\nDivisão final dos dados:")
print(f"Treino: {len(dataset_final['train'])} amostras")
print(f"Validação: {len(dataset_final['validation'])} amostras") 
print(f"Teste: {len(dataset_final['test'])} amostras")

print("\nExemplo de avaliação:")
print("Texto:", dataset_final['train'][0]['text'])
print("Label:", dataset_final['train'][0]['label'])

### Tokenização

In [ ]:
# Tokenização usando BERTimbau (modelo brasileiro)
from transformers import AutoTokenizer

# Usar BERTimbau - modelo BERT específico para português brasileiro
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    """
    Tokeniza o texto das avaliações:
    - Usa BERTimbau treinado especificamente em português brasileiro
    - Limita comprimento máximo para eficiência
    - Padding será feito dinamicamente pelo DataCollator
    """
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=256,  # Reviews podem ser um pouco maiores que amazon
        padding=False  # Padding dinâmico
    )

# Aplicando tokenização
tokenized_datasets = dataset_final.map(tokenize_function, batched=True)

# Removendo coluna de texto original (já temos tokens)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

# Definindo formato PyTorch
tokenized_datasets.set_format("torch")

print("Colunas restantes após processamento:")
print(tokenized_datasets['train'].column_names)

print("\nExemplo tokenizado:")
example = tokenized_datasets['train'][0]
print(f"Input IDs: {example['input_ids'][:20]}...")  # Primeiros 20 tokens
print(f"Attention Mask: {example['attention_mask'][:20]}...")
print(f"Label: {example['label']}")

### Data Collator e Validação Final

In [ ]:
# Data Collator para padding dinâmico
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Data Collator configurado para padding dinâmico")

# Validação final dos dados
print("\nValidação dos dados processados:")
print(f"Tamanho do conjunto de treino: {len(tokenized_datasets['train'])}")
print(f"Tamanho do conjunto de validação: {len(tokenized_datasets['validation'])}")
print(f"Tamanho do conjunto de teste: {len(tokenized_datasets['test'])}")

# Verificar distribuição das classes
from collections import Counter

def count_labels(split):
    labels = [example['label'] for example in tokenized_datasets[split]]
    return Counter(labels)

print("\nDistribuição das classes:")
print("Train:", count_labels('train'))
print("Validation:", count_labels('validation'))
print("Test:", count_labels('test'))

# Verificar se os dados estão no formato correto
sample_batch = data_collator([tokenized_datasets['train'][i] for i in range(4)])
print(f"\nExemplo de batch após Data Collator:")
print(f"Input IDs shape: {sample_batch['input_ids'].shape}")
print(f"Attention Mask shape: {sample_batch['attention_mask'].shape}")
print(f"Labels shape: {sample_batch['labels'].shape}")

## Fine-tuning

## Etapa 3: Fine-tuning do Modelo para Análise de Sentimentos

### Modelo Escolhido: BERTimbau

Utilizaremos o **BERTimbau** (neuralmind/bert-base-portuguese-cased), um modelo BERT pré-treinado especificamente em português brasileiro. Este modelo é ideal porque:

- **Treinado em pt-BR**: Melhor compreensão de nuances da língua portuguesa
- **Arquitetura BERT**: Encoder Transformer poderoso para tarefas de classificação
- **Base sólida**: Pré-treinado em corpus brasileiro, pronto para fine-tuning

### Adaptação para Classificação Binária

Como nosso dataset B2W tem classificação binária (positivo/negativo), adaptaremos:
- `num_labels=2`
- Métricas otimizadas para classificação binária
- Função de perda adequada para 2 classes

## Carregamento do Modelo e Configuração

In [ ]:
# Carregamento do modelo BERTimbau para classificação binária
from transformers import AutoModelForSequenceClassification

model_name = "neuralmind/bert-base-portuguese-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2  # Classificação binária: 0 (negativo) e 1 (positivo)
)

print("Modelo BERTimbau carregado com sucesso!")
print(f"Número de labels: {model.num_labels}")
print(f"Número de parâmetros: {model.num_parameters():,}")

# Verificar se temos GPU disponível
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

# Mover modelo para GPU se disponível
model.to(device)

## Função de Métricas para Classificação Binária

In [ ]:
# Métricas de avaliação para classificação binária
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(eval_pred):
    """
    Computa métricas para avaliação do modelo de sentimento binário
    """
    predictions, labels = eval_pred
    
    # Para classificação binária, pegamos as probabilidades da classe positiva
    preds_proba = predictions[:, 1]  # Probabilidade da classe 1 (positivo)
    preds_binary = np.argmax(predictions, axis=1)  # Predição binária
    
    # Métricas principais
    accuracy = accuracy_score(labels, preds_binary)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds_binary, average='binary', pos_label=1
    )
    
    # AUC-ROC (útil para classificação binária)
    auc_roc = roc_auc_score(labels, preds_proba)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc
    }

print("Função de métricas configurada para classificação binária")

## Configuração do Training Arguments

In [ ]:
# Configuração dos argumentos de treinamento
from transformers import TrainingArguments

output_dir = "./results/sentiment_model"

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,  # Taxa de aprendizado baixa para fine-tuning
    per_device_train_batch_size=16,  # Batch size adequado para BERT
    per_device_eval_batch_size=16,
    num_train_epochs=3,  # 3 épocas para fine-tuning balanceado
    weight_decay=0.01,  # Regularização L2
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy="epoch",  # Avaliação a cada época
    save_strategy="epoch",  # Salvar modelo a cada época
    load_best_model_at_end=True,  # Carregar melhor modelo no final
    metric_for_best_model="f1",  # Usar F1 como métrica principal
    greater_is_better=True,
    fp16=True,  # Mixed precision para acelerar treinamento
    dataloader_pin_memory=False,  # Para compatibilidade
)

print("Training Arguments configurados:")
print(f"- Learning rate: {training_args.learning_rate}")
print(f"- Batch size: {training_args.per_device_train_batch_size}")
print(f"- Epochs: {training_args.num_train_epochs}")
print(f"- Output dir: {training_args.output_dir}")

## Instanciação do Trainer e Treinamento

In [ ]:
# Configuração do Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer configurado com sucesso!")

# Iniciar treinamento
print("\nIniciando treinamento...")
train_result = trainer.train()

print("\nTreinamento concluído!")
print(f"Tempo total: {train_result.metrics['train_runtime']:.2f} segundos")
print(f"Exemplos processados por segundo: {train_result.metrics['train_samples_per_second']:.2f}")

## Avaliação no Conjunto de Teste

### Análise dos Resultados do Fine-tuning

**Métricas de Avaliação para Classificação Binária**:

- **Accuracy**: Proporção de predições corretas
- **Precision**: Dos classificados como positivos, quantos realmente são positivos
- **Recall**: Dos realmente positivos, quantos foram identificados
- **F1-Score**: Média harmônica entre Precision e Recall
- **AUC-ROC**: Capacidade de distinguir entre classes positivas e negativas

**Hiperparâmetros Utilizados**:
- `learning_rate=2e-5`: Taxa baixa para não "esquecer" o conhecimento pré-treinado
- `batch_size=16`: Compromisso entre memória e velocidade
- `epochs=3`: Suficiente para fine-tuning sem overfitting
- `fp16=True`: Acelera treinamento usando precisão mista

**Interpretando os Resultados**:
- F1-score > 0.8: Excelente performance
- F1-score 0.7-0.8: Boa performance  
- F1-score 0.6-0.7: Performance aceitável
- AUC-ROC próximo de 1.0: Excelente separação entre classes

**Possíveis Melhorias** (se tempo permitir):
- Aumentar número de epochs
- Experimentar learning rates diferentes
- Usar técnicas de data augmentation
- Balancear classes se houver desbalanceamento

In [ ]:
# Avaliação final no conjunto de teste
print("Avaliando modelo no conjunto de teste...")

test_results = trainer.evaluate(tokenized_datasets["test"])

print("\nResultados no conjunto de teste:")
for metric, value in test_results.items():
    if metric.startswith('eval_'):
        print(".4f")

# Salvar o melhor modelo
trainer.save_model("./best_sentiment_model")
tokenizer.save_pretrained("./best_sentiment_model")

print("\nModelo salvo em: ./best_sentiment_model")

### Teste Manual do Modelo

In [ ]:
# Teste manual do modelo treinado
from transformers import pipeline

# Carregar modelo treinado
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="./best_sentiment_model",
    tokenizer="./best_sentiment_model",
    device=0 if torch.cuda.is_available() else -1
)

# Exemplos de teste
test_reviews = [
    "Produto excelente, chegou antes do prazo e funciona perfeitamente!",
    "Péssimo atendimento, produto defeituoso e entrega atrasada.",
    "O produto é bom, mas o preço está um pouco alto.",
    "Recomendo muito, qualidade excepcional!",
    "Não comprem, veio com defeito e não consigo trocar."
]

print("Testes manuais do modelo:")
for review in test_reviews:
    result = sentiment_classifier(review)
    sentiment = "Positivo" if result[0]['label'] == 'LABEL_1' else "Negativo"
    confidence = result[0]['score']
    print(f"Review: {review[:50]}...")
    print(f"Predição: {sentiment} (confiança: {confidence:.3f})")
    print("-" * 50)

## Etapa 4: Sumarização de Avaliações e Geração do Relatório

### Integração com Modelo de Sentimentos

Utilizaremos nosso modelo BERTimbau fine-tuned para classificar todas as avaliações, criando uma base sólida para o relatório. Para sumarização, usaremos um modelo pré-treinado em português.

### Modelo de Sumarização: mBART

O **mBART** (facebook/mbart-large-50-many-to-many-mmt) é um modelo Transformer Encoder-Decoder multilingue que:
- Suporta português brasileiro
- Gera resumos concisos e coerentes
- Não precisa de fine-tuning para este projeto (economiza tempo)
- Excelente para tarefas de geração de texto

### Sistema de Relatórios de E-commerce

**Arquitetura do Sistema**:

1. **Classificação de Sentimentos**: Modelo BERTimbau analisa cada avaliação
2. **Agrupamento por Produto**: Estatísticas agregadas por categoria
3. **Sumarização Inteligente**: mBART condensa pontos positivos/negativos
4. **Relatório Estruturado**: Formato legível para gestores

**Aplicações em Engenharia de Dados**:

**Extração**: Pipeline automatizado processa milhares de avaliações

**Transformação**: 
- Classificação de sentimentos
- Agregação por produto/categoria
- Geração de métricas de satisfação

**Loading/Consumo**:
- **APIs**: Relatórios servidos via REST APIs
- **Data Warehouse**: Métricas armazenadas para business intelligence
- **Dashboards**: Power BI/Tableau consomem dados em tempo real
- **Message Queue**: Alertas automáticos via Kafka para produtos com baixa satisfação

**Governança de Dados**:
- **Qualidade**: Validação automática de dados de entrada
- **Conformidade**: Anonimização de dados pessoais
- **Auditoria**: Logs de processamento para rastreabilidade

**Valor de Negócio**:
- **Tempo de Resposta**: De dias para minutos na identificação de problemas
- **Ações Proativas**: Alertas automáticos para produtos defeituosos
- **ROI Mensurável**: Redução de custos com suporte e aumento de vendas

In [ ]:
# Carregamento do modelo de sumarização mBART
from transformers import pipeline
import torch

# Verificar dispositivo disponível
device = 0 if torch.cuda.is_available() else -1

# Carregar pipeline de sumarização
summarizer = pipeline(
    "summarization", 
    model="facebook/mbart-large-50-many-to-many-mmt",
    tokenizer="facebook/mbart-large-50-many-to-many-mmt", 
    device=device
)

print("Modelo de sumarização mBART carregado!")
print(f"Dispositivo: {'GPU' if device == 0 else 'CPU'}")

# Teste rápido do modelo
test_text = "Este produto é excelente, chegou rápido e funciona perfeitamente. Recomendo muito!"
summary = summarizer(test_text, max_length=50, min_length=20, do_sample=False)[0]['summary_text']
print(f"\nTeste de sumarização:")
print(f"Original: {test_text}")
print(f"Resumo: {summary}")

### Carregamento do Modelo de Sentimentos Treinado

In [ ]:
# Carregar modelo de sentimentos treinado
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="./best_sentiment_model",
    tokenizer="./best_sentiment_model", 
    device=device
)

print("Modelo de sentimentos carregado!")

# Teste do classificador
test_reviews = [
    "Produto maravilhoso, entrega rápida!",
    "Péssimo, veio com defeito.",
    "Mais ou menos, preço justo."
]

print("\nTeste do classificador de sentimentos:")
for review in test_reviews:
    result = sentiment_classifier(review)
    label = "Positivo" if result[0]['label'] == 'LABEL_1' else "Negativo"
    score = result[0]['score']
    print(f"'{review}' → {label} ({score:.3f})")

### Sistema de Relatórios por Produto

In [ ]:
# Sistema de geração de relatórios por produto/categoria
import pandas as pd
from collections import defaultdict

class ProductReviewAnalyzer:
    """
    Sistema para analisar avaliações de produtos e gerar relatórios
    """
    
    def __init__(self, sentiment_model, summarizer_model):
        self.sentiment_model = sentiment_model
        self.summarizer = summarizer_model
    
    def analyze_reviews(self, reviews_df, product_column='product_category', 
                       text_column='review_text_processed'):
        """
        Analisa todas as avaliações e retorna estatísticas por produto
        """
        results = defaultdict(lambda: {
            'total_reviews': 0,
            'positive_reviews': [],
            'negative_reviews': [],
            'positive_count': 0,
            'negative_count': 0,
            'sentiment_distribution': {'positive': 0, 'negative': 0}
        })
        
        print(f"Analisando {len(reviews_df)} avaliações...")
        
        for idx, row in reviews_df.iterrows():
            product = row.get(product_column, 'unknown')
            text = row[text_column]
            
            # Classificar sentimento
            sentiment_result = self.sentiment_model(text)
            sentiment = 'positive' if sentiment_result[0]['label'] == 'LABEL_1' else 'negative'
            confidence = sentiment_result[0]['score']
            
            # Armazenar resultados
            results[product]['total_reviews'] += 1
            results[product]['sentiment_distribution'][sentiment] += 1
            
            if sentiment == 'positive':
                results[product]['positive_reviews'].append(text)
                results[product]['positive_count'] += 1
            else:
                results[product]['negative_reviews'].append(text)
                results[product]['negative_count'] += 1
            
            # Progress update a cada 1000 reviews
            if (idx + 1) % 1000 == 0:
                print(f"Processadas {idx + 1} avaliações...")
        
        return dict(results)
    
    def generate_product_report(self, product_name, analysis_results, max_reviews_for_summary=5):
        """
        Gera relatório completo para um produto específico
        """
        if product_name not in analysis_results:
            return f"Produto '{product_name}' não encontrado nos dados."
        
        data = analysis_results[product_name]
        total = data['total_reviews']
        positive_pct = (data['positive_count'] / total) * 100 if total > 0 else 0
        negative_pct = (data['negative_count'] / total) * 100 if total > 0 else 0
        
        report = f"""
# Relatório de Análise de Sentimentos - {product_name}

## Estatísticas Gerais
- **Total de avaliações**: {total}
- **Avaliações positivas**: {data['positive_count']} ({positive_pct:.1f}%)
- **Avaliações negativas**: {data['negative_count']} ({negative_pct:.1f}%)

## Pontos Positivos
"""
        
        # Sumarizar avaliações positivas
        if data['positive_reviews']:
            positive_sample = data['positive_reviews'][:max_reviews_for_summary]
            positive_text = " ".join(positive_sample)
            
            try:
                positive_summary = self.summarizer(
                    positive_text, 
                    max_length=100, 
                    min_length=30, 
                    do_sample=False
                )[0]['summary_text']
                report += f"{positive_summary}\n"
            except:
                report += "Não foi possível gerar resumo dos pontos positivos.\n"
        
        report += "\n## Pontos de Melhoria\n"
        
        # Sumarizar avaliações negativas
        if data['negative_reviews']:
            negative_sample = data['negative_reviews'][:max_reviews_for_summary]
            negative_text = " ".join(negative_sample)
            
            try:
                negative_summary = self.summarizer(
                    negative_text, 
                    max_length=100, 
                    min_length=30, 
                    do_sample=False
                )[0]['summary_text']
                report += f"{negative_summary}\n"
            except:
                report += "Não foi possível gerar resumo dos pontos negativos.\n"
        
        # Recomendações baseadas nos dados
        report += "\n## Recomendações\n"
        if positive_pct > 70:
            report += "- **Produto bem avaliado**: Manter qualidade e padrões de entrega\n"
        elif positive_pct > 50:
            report += "- **Produto aceitável**: Focar em melhorar pontos negativos identificados\n"
        else:
            report += "- **Atenção necessária**: Revisar qualidade do produto e processo de entrega\n"
        
        # Exemplos de avaliações
        report += "\n## Exemplos de Avaliações\n"
        
        report += "\n**Avaliações Positivas:**\n"
        for i, review in enumerate(data['positive_reviews'][:3]):
            report += f"{i+1}. {review[:100]}...\n"
        
        report += "\n**Avaliações Negativas:**\n"
        for i, review in enumerate(data['negative_reviews'][:3]):
            report += f"{i+1}. {review[:100]}...\n"
        
        return report

# Instanciar o analisador
analyzer = ProductReviewAnalyzer(sentiment_classifier, summarizer)
print("Sistema de análise de produtos configurado!")

### Execução da Análise Completa

In [ ]:
# Executar análise completa dos dados
# Usar uma amostra dos dados para demonstração (ajuste conforme necessário)
sample_size = min(5000, len(df))  # Usar até 5000 reviews para demonstração
df_sample = df.sample(n=sample_size, random_state=42)

print(f"Analisando amostra de {sample_size} avaliações...")

# Executar análise
analysis_results = analyzer.analyze_reviews(
    df_sample, 
    product_column='product_category', 
    text_column='review_text_processed'
)

print("Análise concluída!")
print(f"Produtos analisados: {len(analysis_results)}")

# Mostrar estatísticas gerais
total_reviews = sum(data['total_reviews'] for data in analysis_results.values())
total_positive = sum(data['positive_count'] for data in analysis_results.values())
total_negative = sum(data['negative_count'] for data in analysis_results.values())

print("Estatísticas gerais:")
print(f"- Total de avaliações analisadas: {total_reviews}")
print(f"- Avaliações positivas: {total_positive} ({(total_positive/total_reviews)*100:.1f}%)")
print(f"- Avaliações negativas: {total_negative} ({(total_negative/total_reviews)*100:.1f}%)")

# Mostrar top 5 produtos por volume de avaliações
print("Top 5 produtos por volume de avaliações:")
sorted_products = sorted(analysis_results.items(), 
                        key=lambda x: x[1]['total_reviews'], 
                        reverse=True)

for product, data in sorted_products[:5]:
    print(f"- {product}: {data['total_reviews']} avaliações")

### Geração de Relatório para Produto Específico

In [ ]:
# Gerar relatório detalhado para o produto mais avaliado
if sorted_products:
    top_product = sorted_products[0][0]
    print(f"\nGerando relatório para o produto mais avaliado: {top_product}")
    
    report = analyzer.generate_product_report(top_product, analysis_results)
    
    # Salvar relatório em arquivo
    with open(f'report_{top_product.replace(" ", "_")}.md', 'w', encoding='utf-8') as f:
        f.write(report)
    
    print(f"Relatório salvo em: report_{top_product.replace(' ', '_')}.md")
    
    # Mostrar preview do relatório
    print("\n" + "="*50)
    print("PREVIEW DO RELATÓRIO:")
    print("="*50)
    print(report[:500] + "...")
    print("="*50)